In [ ]:
# Databricks notebook source
# MAGIC
# MAGIC **Dataset:** 284,807 credit card transactions with fraud labels
# MAGIC - 28 anonymized features (V1-V28) from PCA transformation
# MAGIC - Time, Amount, and Class (0=normal, 1=fraud)
# MAGIC
# MAGIC **Design:**
# MAGIC - Read CSV from ADLS raw folder
# MAGIC - Land as-is into Bronze Delta table (no transformation)
# MAGIC - Add audit columns (ingestion_ts, ingestion_date, source)
# MAGIC - Partitioned by ingestion_date
# MAGIC - Append-only

In [ ]:
SP_CLIENT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")

STORAGE_ACCOUNT  = "retailbankingdl"
ADLS_BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Source CSV location
RAW_CSV_PATH     = f"{ADLS_BRONZE_PATH}/kaggle_fraud_raw/creditcard.csv"

# Target Delta table
BRONZE_TABLE_PATH = f"{ADLS_BRONZE_PATH}/transactions_raw"

print("✅ Configuration loaded")

In [ ]:
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print("✅ ADLS Gen2 connection configured")

In [ ]:
print("Checking CSV file exists...")

try:
    files = dbutils.fs.ls(f"{ADLS_BRONZE_PATH}/kaggle_fraud_raw/")
    for f in files:
        size_mb = round(f.size / 1024 / 1024, 1)
        print(f"  ✅ Found: {f.name} ({size_mb} MB)")
except Exception as e:
    raise Exception(f"❌ Cannot find CSV file: {str(e)}")

In [ ]:
print("Reading CSV file...")

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RAW_CSV_PATH)
)

row_count = df_raw.count()
col_count = len(df_raw.columns)

print(f"  ✅ Loaded: {row_count:,} rows × {col_count} columns")
print(f"  Columns: {df_raw.columns}")

In [ ]:
print("Sample data (5 rows):")
df_raw.select("Time", "Amount", "Class", "V1", "V2", "V3").show(5)

fraud_count = df_raw.filter(df_raw.Class == 1).count()
normal_count = df_raw.filter(df_raw.Class == 0).count()
fraud_pct = round(fraud_count / row_count * 100, 3)

print(f"\n📊 Class distribution:")
print(f"  Normal transactions (Class=0): {normal_count:,}")
print(f"  Fraud transactions  (Class=1): {fraud_count:,} ({fraud_pct}%)")

In [ ]:
from pyspark.sql.functions import lit, current_timestamp
from datetime import datetime

ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")

df_bronze = (df_raw
    .withColumn("source",         lit("kaggle_creditcard_fraud"))
    .withColumn("ingestion_ts",   current_timestamp())
    .withColumn("ingestion_date", lit(ingestion_date))
)

print("✅ Audit columns added: source, ingestion_ts, ingestion_date")
print(f"   Total columns now: {len(df_bronze.columns)}")

In [ ]:
print(f"Writing to Bronze Delta table...")
print(f"  Target: {BRONZE_TABLE_PATH}")

(df_bronze.write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .save(BRONZE_TABLE_PATH)
)

print(f"✅ Written successfully")

In [ ]:
print("Verifying Bronze Delta table...")

df_verify = spark.read.format("delta").load(BRONZE_TABLE_PATH)

total_rows = df_verify.count()
partitions = df_verify.select("ingestion_date").distinct().collect()
partition_dates = [r["ingestion_date"] for r in partitions]

print(f"  ✅ Total rows in Delta table: {total_rows:,}")
print(f"  ✅ Partitions: {partition_dates}")
print(f"  ✅ Columns: {df_verify.columns}")

df_verify.select("Time", "Amount", "Class", "source", "ingestion_date").show(5)

In [ ]:
print("Checking Delta history...")

spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_TABLE_PATH}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [ ]:
total_rows = spark.read.format("delta").load(BRONZE_TABLE_PATH).count()

print("=" * 60)
print("KAGGLE FRAUD DATASET — BRONZE INGESTION COMPLETE")
print("=" * 60)
print(f"  ✅ Source: kaggle_fraud_raw/creditcard.csv")
print(f"  ✅ Rows loaded: {total_rows:,}")
print(f"  ✅ Format: Delta Lake (append-only)")
print(f"  ✅ Location: {BRONZE_TABLE_PATH}")
print(f"  ✅ Partitioned by: ingestion_date")
print(f"  ✅ Audit columns: source, ingestion_ts, ingestion_date")
print("=" * 60)
print("Bronze layer now has 8 Delta tables.")
print("Ready for Week 2 Day 2 — Silver cleaning.")
print("=" * 60)